# Exercise 2: PyTorch core

In this exercise you’ll build core PyTorch “muscle memory” that you’ll reuse in basically every model you write:

- **Autograd**: how gradients are created, how they accumulate, and how to compute gradients for one or multiple inputs.
- **Dataloading**: writing small `Dataset`s, using `DataLoader`, and custom `collate_fn`.
- **Optimizers**: implementing **AdamW** updates from scratch (state, bias correction, weight decay).
- **Training basics**: a clean single training step.
- **Initialization**: fan-in/out and common initializers (Xavier / Kaiming), plus a helper to init `nn.Linear`.

As before: fill in all `TODO`s without changing function names or signatures.
When debugging, print shapes/dtypes/devices, and write tiny sanity checks (e.g. compare to PyTorch’s built-ins).


In [39]:
from __future__ import annotations
from dataclasses import dataclass
import torch
from torch import nn

## Autograd fundamentals

PyTorch builds a computation graph when you apply operations to tensors with `requires_grad=True`.
Calling `backward()` (or `torch.autograd.grad`) computes gradients by traversing that graph.

### Key concepts
- **Leaf tensor**: a tensor created by you (not the result of an operation) with `requires_grad=True`. Leaf tensors can store gradients in `.grad`.
- **Gradient accumulation**: calling `backward()` adds into `.grad` (it does not overwrite). You must reset gradients between steps/calls.
- **`torch.autograd.grad` vs `.backward()`**
  - `torch.autograd.grad(f, x)` returns `df/dx` directly and does not write into `x.grad` unless you explicitly do so.
  - `f.backward()` writes gradients into `.grad` of leaf tensors.

In the next functions you’ll compute gradients for a simple scalar function such as `f(x) = sum(x^2)` using both APIs.

### `torch.no_grad()`
Wrap inference-only code to avoid tracking gradients and building graphs:
- saves memory
- speeds up evaluation

### `detach()`
`y = x.detach()` returns a tensor that shares data with `x` but is **not connected** to the autograd graph.
This is useful when you want to treat something as a constant target.

### `model.train()` vs `model.eval()`
- `train()` enables training behavior (e.g. dropout active, batchnorm updates running stats).
- `eval()` enables inference behavior (e.g. dropout off, batchnorm uses running stats).

In [40]:
def grad_with_autograd_grad(x: torch.Tensor) -> torch.Tensor:
    """
    Compute gradient of f(x) = sum(x^2) using torch.autograd.grad

    Requirements:
    - Do not call .backward().
    - x should require grad inside the function (don't assume it does).
    - Must return df/dx
    """

    x_in = x.detach().requires_grad_(True)
    f = torch.sum(x_in ** 2)

    grads = torch.autograd.grad(f, x_in)
    return grads[0]
    raise NotImplementedError

In [41]:

def grad_with_backward(x: torch.Tensor) -> torch.Tensor:
    """
    Compute gradient of f(x) = sum(x^2) using .backward().

    Requirements:
    - Must return df/dx
    - Must not leak gradients across calls (watch x.grad accumulation)
    """

    x_in = x.detach().requires_grad_(True)

    f = torch.sum(x_in ** 2)
    f.backward()

    grad = x_in.grad.clone()
    return grad
    raise NotImplementedError

In [42]:
def grad_wrt_multiple_inputs(
    a: torch.Tensor, b: torch.Tensor,
) -> tuple[torch.Tensor, torch.Tensor]:
    """
    Compute gradients w.r.t. multiple inputs. The function is f(a, b) = sum(a^2 + ab).

    Return:
        (df/da, df/db)

    Requirements:
    - Use torch.autograd.grad
    - Ensure both a and b require grad in this function.
    """
    a_in = a.detach().requires_grad_(True)
    b_in = b.detach().requires_grad_(True)

    f = torch.sum(a_in **2 + a_in * b_in)

    grads = torch.autograd.grad(f, (a_in, b_in))
    return grads[0], grads[1]
    raise NotImplementedError

In [43]:
# Test 1: grad_with_autograd_grad
x1 = torch.tensor([1.0, 2.0, 3.0])
print("Autograd Grad result:", grad_with_autograd_grad(x1))
# Expected for f(x) = sum(x^2) -> df/dx = 2*x: tensor([2., 4., 6.])

# Test 2: grad_with_backward
x2 = torch.tensor([1.0, 2.0, 3.0])
print("Backward method result:", grad_with_backward(x2))
# Expected: tensor([2., 4., 6.])

# Test 3: grad_wrt_multiple_inputs
a = torch.tensor([2.0, 3.0])
b = torch.tensor([4.0, 5.0])
da, db = grad_wrt_multiple_inputs(a, b)
print("Multiple inputs - da:", da) # Expected: 2*a + b -> tensor([ 8.0, 11.0 ])
print("Multiple inputs - db:", db) # Expected: a -> tensor([ 2.0,  3.0 ])

Autograd Grad result: tensor([2., 4., 6.])
Backward method result: tensor([2., 4., 6.])
Multiple inputs - da: tensor([ 8., 11.])
Multiple inputs - db: tensor([2., 3.])


## Dataloading

In PyTorch, a `Dataset` defines how to fetch a *single* training example, and a `DataLoader` handles:
- batching
- shuffling
- parallel workers
- optional custom batching logic via `collate_fn`

### `Dataset` in one sentence
A `Dataset` only needs:
- `__len__`: number of items
- `__getitem__`: return one item (e.g. `(x, y)`)

### Why `collate_fn` matters
The default DataLoader collation stacks items along a new batch dimension.
That works for fixed-size tensors, but it breaks for **variable-length sequences**.

So we’ll implement padding ourselves:
- Convert a list of 1D token sequences into a padded tensor `(B, T_max)`
- Track `lengths` and a `padding_mask`

### Mask convention for padding
For padding masks in this exercise:
- `padding_mask[b, t] == True` means **this is padding / invalid**
- `padding_mask[b, t] == False` means **this is a real token**

In [44]:
from torch.utils.data import DataLoader, Dataset

In [45]:
class TensorPairDataset(Dataset):
    """
    Minimal dataset wrapping (x, y).

    x: (N, ...)
    y: (N, ...)

    N is the number of samples. The dataset should return tuples of (x[i], y[i]).
    """

    def __init__(self, x: torch.Tensor, y: torch.Tensor):
        self.x = x
        self.y = y
        

    def __len__(self) -> int:
        return len(self.x)
        raise NotImplementedError

    def __getitem__(self, idx: int) -> tuple[torch.Tensor, torch.Tensor]:
        return self.x[idx], self.y[idx]
        raise NotImplementedError

In [46]:
class NextTokenDataset(Dataset):
    """
    Next-token prediction dataset.

    Given tokens of shape (N, T), produce:
      input_ids  = tokens[:, :-1]
      target_ids = tokens[:, 1:]

    Return per item:
      (input_ids, target_ids)

    Notes:
    - Returned tensors should be 1D of length (T-1).
    - dtype should remain integer.
    """

    def __init__(self, tokens: torch.Tensor):
        self.tokens = tokens
        
    def __len__(self) -> int:
        return len(self.tokens)
        raise NotImplementedError

    def __getitem__(self, idx: int) -> tuple[torch.Tensor, torch.Tensor]:
        row = self.tokens[idx]
        return row[:-1], row[1:]
        raise NotImplementedError

In [47]:

class RandomCropSequenceDataset(Dataset):
    """
    Sequence dataset that returns random crops of fixed length.

    tokens: (N, T_total)
    crop_len: L

    For each __getitem__:
      - sample a start index s so that s+L <= T_total
      - return tokens[idx, s:s+L]

    Requirements:
    - Use a torch.Generator for deterministic behavior if seed is provided.
    - Do NOT use Python's random module.
    """

    def __init__(self, tokens: torch.Tensor, crop_len: int, seed: int | None = None):
        self.tokens = tokens
        self.crop_len = crop_len
        self.N, self.T_total = tokens.shape

        self.generator = torch.Generator() 
        if seed is not None:
            self.generator.manual_seed(seed)
        raise NotImplementedError

    def __len__(self) -> int:
        return self.N
        raise NotImplementedError

    def __getitem__(self, idx: int) -> torch.Tensor:
        max_start = self.T_total - self.crop_len
        start = torch.randint(0, max_start + 1, (1,), generator=self.generator).item()
        return self.tokens[dx, start : start + self.crop_len]
        raise NotImplementedError

In [48]:


@dataclass(frozen=True)
class PaddedBatch:
    """
    A padded batch for variable-length sequences.

    tokens: LongTensor (B, T_max)
    lengths: LongTensor (B,)
    padding_mask: BoolTensor (B, T_max) where True means "this is padding"
    """

    tokens: torch.Tensor
    lengths: torch.Tensor
    padding_mask: torch.Tensor


def pad_1d_sequences(seqs: list[torch.Tensor], pad_value: int = 0) -> PaddedBatch:
    """
    Pad a list of 1D integer tensors to the same length.

    Requirements:
    - Return PaddedBatch(tokens, lengths, padding_mask)
    - padding_mask[b, t] == True iff t >= lengths[b]
    - tokens should be dtype long, if not cast them
    """
    lengths = torch.tensor([len(s) for s in seqs], dtype=torch.long)
    T_max = lengths.max().item() if len(lengths) > 0 else 0
    
    # Ensure tokens are dtype long
    seqs_long = [s.long() for s in seqs]
    
    # Pad sequences
    tokens = torch.nn.utils.rnn.pad_sequence(seqs_long, batch_first=True, padding_value=pad_value)
    
    # padding_mask[b, t] == True iff t >= lengths[b]
    ranges = torch.arange(T_max, device=tokens.device).unsqueeze(0) # (1, T_max)
    padding_mask = ranges >= lengths.unsqueeze(1) # (B, T_max)
    
    return PaddedBatch(tokens=tokens, lengths=lengths, padding_mask=padding_mask)
    raise NotImplementedError

In [49]:
def collate_next_token_batch(
    batch: list[tuple[torch.Tensor, torch.Tensor]], pad_value: int = 0
) -> dict[str, torch.Tensor]:
    """
    Collate for NextTokenDataset samples that may have variable lengths.

    batch: list of (input_ids, target_ids), each 1D

    Return dict with:
      - input_ids: (B, T_max)
      - target_ids: (B, T_max)
      - attention_mask: (B, T_max) where True means "keep" (NOT padding)
      - padding_mask: (B, T_max) where True means "padding"

    Requirements:
    - pad input_ids and target_ids consistently
    - attention_mask is the logical NOT of padding_mask
    """
    input_seqs = [item[0] for item in batch]
    target_seqs = [item[1] for item in batch]
    
    padded_inputs = pad_1d_sequences(input_seqs, pad_value=pad_value)
    padded_targets = pad_1d_sequences(target_seqs, pad_value=pad_value)
    
    # attention_mask is logical NOT of padding_mask
    attention_mask = ~padded_inputs.padding_mask
    
    return {
        "input_ids": padded_inputs.tokens,
        "target_ids": padded_targets.tokens,
        "attention_mask": attention_mask,
        "padding_mask": padded_inputs.padding_mask
    }
    raise NotImplementedError

In [50]:
def make_dataloader(
    dataset: Dataset,
    batch_size: int,
    shuffle: bool = True,
    drop_last: bool = False,
    collate_fn=None,
    num_workers: int = 0,
) -> DataLoader:
    """
    Create a DataLoader with optional collate_fn.
    """
    return DataLoader(
        dataset=dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        collate_fn=collate_fn,
        num_workers=num_workers
    )
    raise NotImplementedError

In [51]:
# 1. Test NextTokenDataset and Padding
tokens = torch.tensor([[10, 20, 30, 40, 50], 
                       [15, 25, 35, 0, 0]]) # Second sequence shorter conceptually (length 3)
dataset = NextTokenDataset(tokens)
input_ids, target_ids = dataset[0]

print("Single item input_ids:", input_ids)
print("Single item target_ids:", target_ids)

# 2. Test collate function
collate_batch = [dataset[0], dataset[1]]
collated = collate_next_token_batch(collate_batch)
print("\nCollated batch keys:", collated.keys())
print("Padded input_ids shape:", collated["input_ids"].shape)
print("Padding mask:\n", collated["padding_mask"])
print("Attention mask:\n", collated["attention_mask"])

Single item input_ids: tensor([10, 20, 30, 40])
Single item target_ids: tensor([20, 30, 40, 50])

Collated batch keys: dict_keys(['input_ids', 'target_ids', 'attention_mask', 'padding_mask'])
Padded input_ids shape: torch.Size([2, 4])
Padding mask:
 tensor([[False, False, False, False],
        [False, False, False, False]])
Attention mask:
 tensor([[True, True, True, True],
        [True, True, True, True]])


## Optimizers (AdamW from scratch)

PyTorch optimizers keep **state** for each parameter (e.g. moment estimates in Adam).
In this section you’ll implement **AdamW**, which is Adam + *decoupled* weight decay.

### AdamW state
For each parameter tensor `p` we store:
- `m`: first moment (EMA of gradients)
- `v`: second moment (EMA of squared gradients)
- `t`: step counter

### Update overview (high level)
1) Update moments `m, v`
2) Bias-correct them (`m_hat, v_hat`)
3) Apply parameter update:
   `p -= lr * ( m_hat / (sqrt(v_hat) + eps) + weight_decay * p )`

Notes:
- This update is **in-place** (mutates `p`).
- Gradients should not be modified.
- State tensors must match parameter shape/device/dtype.

In [52]:
@dataclass
class AdamWState:
    """
    Per-parameter AdamW state.

    m: first moment
    v: second moment
    t: step count
    """

    m: torch.Tensor
    v: torch.Tensor
    t: int


def init_adamw_state(p: torch.Tensor) -> AdamWState:
    """
    Initialize AdamW state tensors for a parameter tensor p.

    What to create:
    - m: zeros like p, same shape/device/dtype
    - v: zeros like p, same shape/device/dtype
    - t: step counter starting at 0

    Notes / requirements:
    - Use torch.zeros_like(p) for m and v.
    - Do NOT attach gradients to the state (initialize under torch.no_grad()).
    - t starts at 0. In adamw_step_, increment t to 1 on the first update *before*
      computing bias correction terms (1 - beta1^t) and (1 - beta2^t).
    - State tensors must live on the same device as p (CPU vs GPU) and have the
      same dtype as p.
    """
    with torch.no_grad():
      m = torch.zeros_like(p)
      v = torch.zeros_like(p)
    return AdamWState(m=m, v=v, t=0)   

In [61]:
def adamw_step_(
    p: torch.Tensor,
    grad: torch.Tensor,
    state: AdamWState,
    lr: float,
    betas: tuple[float, float] = (0.9, 0.999),
    eps: float = 1e-8,
    weight_decay: float = 0.01,
) -> AdamWState:
    """
    In-place AdamW parameter update (updates p).

    Algorithm (AdamW):
      m = beta1*m + (1-beta1)*grad
      v = beta2*v + (1-beta2)*grad^2
      m_hat = m / (1 - beta1^t)
      v_hat = v / (1 - beta2^t)
      p = p - lr * (m_hat / (sqrt(v_hat) + eps) + weight_decay * p)

    Requirements:
    - Update p in-place.
    - Return updated state (with incremented t).
    - Do not modify grad.
    - Should work for any tensor shape.
    """
    beta1, beta2 = betas
    
    # 1. Increment step count t first (before bias correction)
    t = state.t + 1
    
    # 2. Update biased first and second moments (do not modify grad)
    m = beta1 * state.m + (1 - beta1) * grad
    v = beta2 * state.v + (1 - beta2) * (grad ** 2)
    
    # 3. Compute bias-corrected moments
    m_hat = m / (1 - (beta1 ** t))
    v_hat = v / (1 - (beta2 ** t))
    
    # 4. In-place update of parameter p (incorporating weight decay and update step)
    with torch.no_grad():
        p.sub_(lr * (m_hat / (torch.sqrt(v_hat) + eps) + weight_decay * p))
        
    return AdamWState(m=m, v=v, t=t)

In [54]:
def adamw_step_many_(
    params: list[torch.Tensor],
    grads: list[torch.Tensor],
    states: list[AdamWState],
    lr: float,
    betas: tuple[float, float] = (0.9, 0.999),
    eps: float = 1e-8,
    weight_decay: float = 0.01,
) -> list[AdamWState]:
    """
    Apply AdamW to many parameters.

    Requirements:
    - len(params) == len(grads) == len(states)
    - Update each param in-place.
    - Return the list of updated states.
    """
    assert len(params) == len(grads) == len(states)
    updated_states = []
    for p, g, s in zip(params, grads, states):
        new_state = adamw_step(p, g, s, lr=lr, betas=betas, eps=eps, weight_decay=weight_decay)
        updated_states.append(new_state)
    return updated_states

## Training basics

A minimal training step follows the same pattern almost everywhere:

1) set model to train mode
2) reset gradients
3) forward pass
4) compute loss
5) backward pass
6) step optimizer

In this exercise you’ll implement a single MSE training step using a standard PyTorch optimizer.
Return a Python float loss value.

In [55]:
def train_step_mse(
    model: nn.Module,
    batch: tuple[torch.Tensor, torch.Tensor],
    optimizer: torch.optim.Optimizer,
) -> float:
    """
    One MSE train step using standard torch optimizer.
    """
    optimizer.zero_grad()
    x, y = batch
    preds = model(x)
    loss = torch.nn.functional.mse_loss(preds, y)
    loss.backward()
    optimizer.step()
    return loss.item()


## Parameter initialization

Initialization matters because it controls signal and gradient scales at the start of training.

### Fan-in / fan-out
- `fan_in`: number of input connections to a unit
- `fan_out`: number of output connections from a unit

For a Linear layer weight of shape `(out_features, in_features)`:
- `fan_in = in_features`
- `fan_out = out_features`

### Common schemes
- **Xavier / Glorot** (often good for tanh / linear-ish nets):
  keeps variance stable across layers when activations are roughly symmetric.
- **Kaiming / He** (often good for ReLU-like nets):
  accounts for the fact that ReLU zeroes out about half the inputs.

In this section you’ll implement Xavier uniform and Kaiming uniform and use them to initialize `nn.Linear`.
We also always zero the bias unless explicitly told otherwise.

In [56]:
def fan_in_fan_out(weight: torch.Tensor) -> tuple[int, int]:
    """Compute (fan_in, fan_out) for a weight tensor."""
    dimensions = weight.dim()
    if dimensions < 2:
        raise ValueError("Fan in and fan out can not be computed for tensor with fewer than 2 dimensions")
    
    # For Linear layer weights of shape (out_features, in_features):
    # fan_in is in_features (dim 1), fan_out is out_features (dim 0)
    fan_out = weight.size(0)
    fan_in = weight.size(1)
    
    # For higher dimensional convolutions (e.g., Conv2d: out_channels, in_channels, kH, kW)
    if dimensions > 2:
        receptive_field_size = weight[0][0].numel()
        fan_in *= receptive_field_size
        fan_out *= receptive_field_size
        
    return fan_in, fan_out

In [57]:
def xavier_uniform_(weight: torch.Tensor, gain: float = 1.0) -> torch.Tensor:
    """
    In-place Xavier/Glorot uniform init:
      bound = gain * sqrt(6 / (fan_in + fan_out))
      U(-bound, bound)
    """
    fan_in, fan_out = fan_in_fan_out(weight)
    bound = gain * math.sqrt(6.0 / (fan_in + fan_out))
    return weight.data.uniform_(-bound, bound)

In [64]:
def kaiming_uniform_(weight: torch.Tensor, nonlinearity: str = "relu") -> torch.Tensor:
    """
    In-place Kaiming/He uniform init.

    Follow this common choice:
      gain = sqrt(2) for ReLU
      std = gain / sqrt(fan_in)
      bound = sqrt(3) * std
      U(-bound, bound)
    """
    fan_in, _ = fan_in_fan_out(weight)
    
    # Gain calculation based on nonlinearity
    gain = math.sqrt(2.0) if nonlinearity == "relu" else 1.0
    std = gain / math.sqrt(fan_in)
    bound = math.sqrt(3.0) * std
    return weight.data.uniform_(-bound, bound)

In [65]:
def init_linear_(layer: nn.Linear, scheme: str = "xavier") -> nn.Linear:
    """
    Initialize an nn.Linear in-place.

    scheme:
      - "xavier"
      - "kaiming_relu"
      - "zero" (weights and bias = 0)
    """
    if scheme == "xavier":
        xavier_uniform_(layer.weight, gain=1.0)
    elif scheme == "kaiming_relu":
        kaiming_uniform_(layer.weight, nonlinearity="relu")
    elif scheme == "zero":
        layer.weight.data.zero_()
    else:
        raise ValueError(f"Unknown initialization scheme: {scheme}")
        
    if layer.bias is not None:
        layer.bias.data.zero_()
        
    return layer

In [66]:
import torch
import torch.nn as nn
import math

# 1. Test AdamW initialization and step
p = torch.tensor([1.0, 2.0], requires_grad=True)
grad = torch.tensor([0.1, 0.2])
state = init_adamw_state(p)
state = adamw_step_(p, grad, state, lr=0.001)
print("Updated parameter p:", p)
print("Updated step count t:", state.t)

# 2. Test Linear Layer Initialization
linear = nn.Linear(5, 3)
init_linear_(linear, scheme="kaiming_relu")
print("Initialized weight shape:", linear.weight.shape)
print("Bias is zero:", torch.all(linear.bias == 0))

Updated parameter p: tensor([0.9990, 1.9990], requires_grad=True)
Updated step count t: 1
Initialized weight shape: torch.Size([3, 5])
Bias is zero: tensor(True)
